# Перевірка знань: API та REST

**20 питань.** Notebook складається з трьох частин:

- **Частина 1 (1–8):** теоретичні питання — напишіть відповідь у клітинці нижче питання.
- **Частина 2 (9–15):** знайдіть помилку — поясніть, у чому проблема, і напишіть виправлений код у порожній клітинці.
- **Частина 3 (16–20):** що виведе код — спочатку напишіть свою відповідь, **не запускаючи** клітинку, потім перевірте себе.

Під кожним питанням є згорнутий блок **«💡 Відповідь»** — відкривайте його лише після того, як дали власну відповідь!

> Питання охоплюють: поняття API та REST, HTTP-методи й CRUD, статус-коди, query- та path-параметри, заголовки й тіло запиту, бібліотеку `requests` (`params`/`json`/`timeout`/`raise_for_status`), автентифікацію (Bearer-токен), створення власного API на FastAPI (маршрути, валідація через Pydantic, `HTTPException`) та асинхронні запити `httpx`.

> Перед роботою виконайте клітинку нижче.

Успіхів! 🍀

In [1]:
import requests, httpx, pydantic, json

print("requests:", requests.__version__)
print("httpx:   ", httpx.__version__)
print("pydantic:", pydantic.VERSION)
print("Готово до роботи ✅")

requests: 2.33.1
httpx:    0.28.1
pydantic: 2.13.4
Готово до роботи ✅


---
## Частина 1. Теорія

### Питання 1
Що таке **API** і що таке **REST**? У якому форматі зазвичай передаються дані в REST-API?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<b>API</b> (Application Programming Interface) — це «контракт», за яким одна програма звертається до іншої. <b>REST</b> — це поширений стиль веб-API, де ресурси доступні за URL, а дії над ними виконуються звичайними <b>HTTP-методами</b> (<code>GET</code>, <code>POST</code>, ...). Дані передаються здебільшого у форматі <b>JSON</b> — текстовому, незалежному від мови програмування.

</details>

### Питання 2
Перелічіть основні **HTTP-методи** REST-API та вкажіть, яким операціям **CRUD** вони відповідають.

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<ul>
<li><code>GET</code> — прочитати дані (<b>R</b>ead);</li>
<li><code>POST</code> — створити новий ресурс (<b>C</b>reate);</li>
<li><code>PUT</code> / <code>PATCH</code> — оновити (повністю / частково) (<b>U</b>pdate);</li>
<li><code>DELETE</code> — видалити (<b>D</b>elete).</li>
</ul>Тобто <b>CRUD</b> = Create / Read / Update / Delete.

</details>

### Питання 3
Що означають класи статус-кодів **2xx**, **4xx** та **5xx**? Наведіть приклади кодів `200`, `201`, `404`, `500`.

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<ul>
<li><b>2xx — успіх:</b> <code>200 OK</code> (успішно), <code>201 Created</code> (ресурс створено), <code>204 No Content</code> (успішно, тіла немає);</li>
<li><b>4xx — помилка клієнта:</b> <code>400 Bad Request</code> (погані дані), <code>401 Unauthorized</code> (немає автентифікації), <code>404 Not Found</code> (ресурс не знайдено);</li>
<li><b>5xx — помилка сервера:</b> <code>500 Internal Server Error</code> (збій на стороні сервера).</li>
</ul>Коротко: <b>4xx</b> — «винен клієнт», <b>5xx</b> — «винен сервер».

</details>

### Питання 4
Чим **query-параметр** відрізняється від **path-параметра**? Наведіть приклад URL для кожного.

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<ul>
<li><b>Path-параметр</b> — частина самого шляху, ідентифікує конкретний ресурс: <code>/users/<b>7</b></code>;</li>
<li><b>Query-параметр</b> — після <code>?</code>, для фільтрації/сортування/сторінкування: <code>/users<b>?role=admin&page=2</b></code>.</li>
</ul>У <code>requests</code> query-параметри зручно передавати словником у <code>params=</code>. У FastAPI path-параметр оголошують у шляху <code>/users/{user_id}</code>, а query — як аргумент функції зі значенням за замовчуванням.

</details>

### Питання 5
Як у бібліотеці `requests` надіслати **JSON-тіло** у `POST`-запиті? Чому зручніше використовувати `json=`, а не `data=`?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<pre><code>requests.post(url, json={"name": "Ivan", "age": 20})</code></pre>Параметр <code>json=</code> сам <b>серіалізує</b> словник у JSON-рядок і автоматично проставляє заголовок <code>Content-Type: application/json</code>. Параметр <code>data=</code> натомість надсилає дані як <b>form-urlencoded</b> (як HTML-форма) і не ставить JSON-заголовок, тож сервер, що очікує JSON, не зрозуміє запит.

</details>

### Питання 6
Навіщо у запитах задавати `timeout=` і викликати `resp.raise_for_status()`?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<ul>
<li><code>timeout=</code> обмежує час очікування: без нього запит може «зависнути» назавжди, якщо сервер не відповідає;</li>
<li><code>resp.raise_for_status()</code> кидає виняток (<code>HTTPError</code>) на статус-кодах <code>4xx</code>/<code>5xx</code>, тож помилку сервера не пропустиш мовчки.</li>
</ul>Разом із <code>try/except requests.exceptions.RequestException</code> це робить мережевий код надійним.

</details>

### Питання 7
Як у запиті передається автентифікація через **Bearer-токен**? Чому ключ/токен не можна писати прямо в коді?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

Токен передають у заголовку <code>Authorization</code> зі схемою <code>Bearer</code>:<br>
<pre><code>headers = {"Authorization": f"Bearer {token}"}
requests.get(url, headers=headers)</code></pre>
Саме так викликають і LLM-провайдерів (OpenAI, Anthropic). Ключ <b>не пишуть у коді</b>, бо код потрапляє в git/репозиторій і ключ може витекти. Його тримають у <b>змінних оточення</b> (<code>os.environ</code>) або у <code>.env</code>-файлі.

</details>

### Питання 8
Що таке **FastAPI** і як він використовує **Pydantic** для обробки тіла запиту?

✍️ *Ваша відповідь:*


<details>
<summary>💡 Відповідь</summary>

<b>FastAPI</b> — сучасний Python-фреймворк для створення REST-API. Маршрути описують декораторами (<code>@app.get</code>, <code>@app.post</code>, ...). Якщо аргумент функції-обробника анотований <b>Pydantic-моделлю</b>, FastAPI автоматично <b>читає тіло запиту, валідує його</b> за моделлю і, якщо дані погані, повертає зрозумілу помилку <code>422</code> — без жодного ручного коду. Крім того, FastAPI генерує інтерактивну документацію (<code>/docs</code>) та OpenAPI-схему.

</details>

---
## Частина 2. Знайди помилку

У кожному фрагменті коду є щонайменше одна помилка. Поясніть, у чому вона полягає, і напишіть виправлений код.

### Питання 9
Сервер очікує JSON, але отримує дані «не в тому форматі» і відповідає `400`. Що не так із відправкою тіла?

In [ ]:
import requests

new_user = {"name": "Ivan", "age": 20}
resp = requests.post("https://httpbingo.org/post", data=new_user)   # ?
print(resp)

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

Параметр <code>data=</code> надсилає словник як <b>form-urlencoded</b> і не ставить заголовок <code>Content-Type: application/json</code>, тож сервер, що чекає JSON, його не приймає. Для JSON-тіла потрібен <code>json=</code>:<br>
<pre><code>resp = requests.post("https://httpbingo.org/post", json=new_user, timeout=10)</code></pre>
Тоді словник серіалізується в JSON і заголовок виставляється автоматично.

</details>

### Питання 10
Після успішного `POST` (сервер повернув `201`) код вважає запит невдалим. Чому перевірка хибна?

In [ ]:
import requests

resp = requests.post("https://httpbingo.org/status/201", timeout=10)
if resp.status_code == 200:        # ?
    print("Створено!")
else:
    print("Помилка:", resp.status_code)

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

Успішне <b>створення</b> ресурсу повертає <code>201 Created</code>, а не <code>200</code>, тож жорстка перевірка <code>== 200</code> хибно вважає запит невдалим. Перевіряйте <b>весь клас 2xx</b> через <code>resp.ok</code> або <code>raise_for_status()</code>:<br>
<pre><code>if resp.ok:                 # True для будь-якого 2xx
    print("Створено!")
# або:
resp.raise_for_status()     # кине виняток лише на 4xx/5xx</code></pre>

</details>

### Питання 11
Хотіли прочитати поле з відповіді, але код кидає `TypeError: 'method' object is not subscriptable`. Що забули?

In [ ]:
import requests

resp = requests.get("https://jsonplaceholder.typicode.com/posts/1", timeout=10)
data = resp.json        # ?
print(data["title"])

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

<code>resp.json</code> — це <b>метод</b>, його треба <b>викликати</b> з дужками, щоб отримати словник. Без дужок <code>data</code> — це сам метод, який не можна індексувати:<br>
<pre><code>data = resp.json()      # дужки!
print(data["title"])</code></pre>

</details>

### Питання 12
У FastAPI хотіли приймати JSON-тіло, але FastAPI вимагає `name`/`age` як query-параметри. Чому?

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class User(BaseModel):
    name: str
    age: int

@app.post("/users")
def create_user(user):        # ?
    return {"created": user}

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

Аргумент <code>user</code> <b>не анотований типом</b>, тож FastAPI не знає, що це тіло запиту, і трактує його як набір query-параметрів. Треба вказати тип — <b>Pydantic-модель</b>:<br>
<pre><code>@app.post("/users")
def create_user(user: User):      # тип -> тіло запиту + валідація
    return {"created": user.model_dump()}</code></pre>
Саме анотація Pydantic-моделлю вмикає читання й валідацію JSON-тіла.

</details>

### Питання 13
Endpoint має повертати `404`, коли ресурсу немає, але клієнт усе одно отримує `200` зі «дивним» тілом. Що не так?

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI()
DB = {1: "Перша нотатка"}

@app.get("/notes/{note_id}")
def get_note(note_id: int):
    if note_id not in DB:
        return HTTPException(status_code=404, detail="Не знайдено")   # ?
    return {"note": DB[note_id]}

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

<code>HTTPException</code> треба <b>підняти</b> (<code>raise</code>), а не <b>повернути</b> (<code>return</code>). Повернутий об'єкт FastAPI просто серіалізує як звичайне тіло зі статусом <code>200</code>. Правильно:<br>
<pre><code>if note_id not in DB:
    raise HTTPException(status_code=404, detail="Не знайдено")</code></pre>
Тоді FastAPI віддасть відповідь зі статусом <code>404</code> і потрібним повідомленням.

</details>

### Питання 14
Захищений endpoint повертає `401 Unauthorized`, хоча токен ніби передали. Що не так із заголовком?

In [ ]:
import requests, os

token = os.environ.get("API_TOKEN", "secret-123")
headers = {"Authorization": token}        # ?
resp = requests.get("https://httpbingo.org/bearer", headers=headers, timeout=10)
print(resp.status_code)

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

Для Bearer-автентифікації значення заголовка має містити <b>схему</b> <code>Bearer</code> перед токеном, а не лише сам токен:<br>
<pre><code>headers = {"Authorization": f"Bearer {token}"}</code></pre>
Без префікса <code>Bearer&nbsp;</code> сервер не розпізнає токен і відповідає <code>401</code>.

</details>

### Питання 15
Асинхронна функція замість даних повертає `<coroutine object>` і сипле попередженням «coroutine was never awaited». Чого бракує?

In [ ]:
import httpx

async def get_title(post_id):
    async with httpx.AsyncClient() as client:
        resp = client.get(f"https://jsonplaceholder.typicode.com/posts/{post_id}")  # ?
        return resp.json()["title"]

✍️ *У чому помилка?*


In [ ]:
# Виправлений код:


<details>
<summary>💡 Відповідь</summary>

Виклик асинхронного методу <code>client.get(...)</code> повертає <b>корутину</b>, яку треба <b>дочекатися</b> через <code>await</code>:<br>
<pre><code>resp = await client.get(f"https://jsonplaceholder.typicode.com/posts/{post_id}")
return resp.json()["title"]</code></pre>
Без <code>await</code> запит фактично не виконується, а <code>resp</code> — це сама корутина, а не відповідь.

</details>

---
## Частина 3. Що виведе код?

Спочатку запишіть свою відповідь у клітинці *«Ваша відповідь»*, і лише потім запустіть код і перевірте себе. Якщо помилилися — поясніть собі, чому код працює інакше.

> ⚠️ Питання 16–20 не потребують інтернету: дані задано в коді або використовуються офлайн-інструменти. Клітинки можна запускати для самоперевірки.

### Питання 16

In [ ]:
def http_class(code):
    if 200 <= code < 300:
        return "success"
    if 400 <= code < 500:
        return "client error"
    if 500 <= code < 600:
        return "server error"
    return "other"

for c in (200, 201, 404, 500):
    print(c, "->", http_class(c))

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>200 -> success
201 -> success
404 -> client error
500 -> server error</code></pre>
<code>200</code> і <code>201</code> належать до класу <b>2xx</b> (успіх), <code>404</code> — до <b>4xx</b> (помилка клієнта), <code>500</code> — до <b>5xx</b> (помилка сервера).

</details>

### Питання 17

In [ ]:
import requests

# .prepare() будує фінальний запит БЕЗ відправки в мережу
req = requests.Request("GET", "https://api.site.com/search",
                       params={"q": "rest api", "page": 2})
print(req.prepare().url)

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>https://api.site.com/search?q=rest+api&page=2</code></pre>
<code>requests</code> сам збирає query-рядок зі словника <code>params</code>: пари <code>ключ=значення</code> з'єднує через <code>&</code>, а спецсимволи екранує — тут пробіл у <code>"rest api"</code> стає <code>+</code>. Числа перетворюються на рядки (<code>page=2</code>).

</details>

### Питання 18

In [ ]:
from pydantic import BaseModel

# Так FastAPI валідує тіло запиту: рядок "20" приводиться до int
class User(BaseModel):
    name: str
    age: int

user = User.model_validate({"name": "Ivan", "age": "20"})
print(user.age, type(user.age).__name__)
print(user.model_dump())

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>20 int
{'name': 'Ivan', 'age': 20}</code></pre>
Pydantic (а отже й FastAPI) <b>розумно приводить типи</b>: рядок <code>"20"</code> стає числом <code>int</code> <b>20</b>. <code>model_dump()</code> повертає звичайний словник із уже приведеними значеннями.

</details>

### Питання 19

In [ ]:
import httpx

# MockTransport імітує сервер локально, без мережі
def handler(request):
    return httpx.Response(200, json={"path": request.url.path})

client = httpx.Client(transport=httpx.MockTransport(handler))
resp = client.get("https://example.com/users/7")
print(resp.status_code)
print(resp.json()["path"])

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>200
/users/7</code></pre>
<code>MockTransport</code> підміняє реальний сервер: на будь-який запит <code>handler</code> повертає відповідь зі статусом <code>200</code> і тілом <code>{"path": ...}</code>, де <code>request.url.path</code> — це шлях запиту, тобто <code>/users/7</code> (без домену та схеми).

</details>

### Питання 20

In [ ]:
import json

# Типовий "round-trip" тіла запиту: dict -> JSON-рядок -> назад
payload = {"name": "Ivan", "active": True, "roles": ["admin", "user"]}

body = json.dumps(payload)
parsed = json.loads(body)

print(type(body).__name__)
print(parsed["roles"][1])
print(parsed == payload)

✍️ *Ваша відповідь (що буде виведено і чому):*


<details>
<summary>💡 Відповідь</summary>

<pre><code>str
user
True</code></pre>
<code>json.dumps</code> перетворює словник на <b>рядок</b> (<code>str</code>) — саме він іде в тілі HTTP-запиту. Після <code>json.loads</code> отримуємо рівнозначний словник, тож <code>parsed == payload</code> → <b>True</b>, а <code>parsed["roles"][1]</code> — другий елемент списку, «<b>user</b>».

</details>